# Витрина эквайринга по атрибутам MVP первой волны

Зачем нужна тетрадка:
- коллега задает месяц и запускает ячейки сверху вниз;
- на выходе получает готовую витрину в CSV для дашборда.

Ниже — расшифровка полей витрины на русском.

In [ ]:
import pandas as pd

attributes_dict_df = pd.DataFrame([
    {'attribute': 'report_month', 'description_ru': 'Отчетный месяц витрины (первый день месяца)'},
    {'attribute': 'inn', 'description_ru': 'ИНН клиента'},
    {'attribute': 'company_name', 'description_ru': 'Наименование клиента'},
    {'attribute': 'agr_id', 'description_ru': 'ID договора из Alpha (если есть)'},
    {'attribute': 'n_agr', 'description_ru': 'Внутренний номер договора в Alpha'},
    {'attribute': 'contract_number', 'description_ru': 'Номер договора в Alpha'},
    {'attribute': 'd_valid_from', 'description_ru': 'Дата начала действия договора'},
    {'attribute': 'd_valid_to', 'description_ru': 'Дата окончания действия договора'},
    {'attribute': 'cdi_id', 'description_ru': 'Идентификатор клиента в CDI'},
    {'attribute': 'ssp_ocrm', 'description_ru': 'Сегмент/блок ОКРМ'},
    {'attribute': 'ogrn', 'description_ru': 'ОГРН клиента'},
    {'attribute': 'filial_rf', 'description_ru': 'Филиал РФ'},
    {'attribute': 'vsp_name', 'description_ru': 'Наименование ВСП'},
    {'attribute': 'vsp_code', 'description_ru': 'Код ВСП'},
    {'attribute': 'tariff_name', 'description_ru': 'Тарифный план'},
    {'attribute': 'retl_cnt', 'description_ru': 'Количество торговых точек'},
    {'attribute': 'term_cnt', 'description_ru': 'Количество терминалов'},
    {'attribute': 'active_term_cnt', 'description_ru': 'Количество активных терминалов (>=1 trx с суммой > 1 за месяц)'},
    {'attribute': 'trx_cnt', 'description_ru': 'Количество операций за месяц'},
    {'attribute': 'trx_sum', 'description_ru': 'Сумма операций за месяц'},
    {'attribute': 'commission_from_ops', 'description_ru': 'Комиссия с операций'},
    {'attribute': 'commission_monthly', 'description_ru': 'Фиксированная комиссия в месяц'},
    {'attribute': 'int_component', 'description_ru': 'Interchange / внутренняя составляющая'},
    {'attribute': 'commission_total', 'description_ru': 'Итоговая комиссия'},
    {'attribute': 'aur', 'description_ru': 'АУР (расчет по терминалам)'},
    {'attribute': 'amortization', 'description_ru': 'Амортизация'},
    {'attribute': 'chod', 'description_ru': 'ЧОД'},
    {'attribute': 'fin_result', 'description_ru': 'Финансовый результат'},
])

attributes_dict_df

## 1) Загрузка библиотек

Подключаем только необходимые библиотеки для расчета и выгрузки.

In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

## 2) Подключение к Impala

Создаем подключение к хранилищу для SQL-запросов.

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

## 3) Параметры запуска

Меняем только `report_month` (первый день нужного месяца).
`month_start` и `month_end` считаются автоматически.

Перед расчетом выполняем `invalidate metadata` по источникам, чтобы снизить риск падений из-за устаревших метаданных Impala.

In [ ]:
# Единственный параметр месяца (первый день месяца)
report_month = '2026-05-01'

# Путь для итогового CSV (предпочтительно для загрузки в Greenplum/DRP)
output_csv_path = './datamart_month_acquiring_2026_05.csv'

report_month_ts = pd.to_datetime(report_month)
month_start = report_month_ts.strftime('%Y-%m-%d')
month_end = (report_month_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
report_month_label = report_month_ts.strftime('%Y-%m')

# Дополнительная техническая метка среза для BI-фильтрации
snapshot_month_start = month_start

# Список источников для обновления метаданных перед запросами
invalidate_tables = [
    'ods_alpha.scd1_agreements',
    'ods_alpha.scd1_companies',
    'ocrm_ul.s_org_ext',
    'cdiul.ext_id_org',
    'ods_alpha.scd1_merchants',
    'ods_alpha.scd1_pos_terminals',
    'sandbox_ai.shestopalov_terminal_amortization_oneoff',
    'ods_alpha.scd1_trx',
    'ods_alpha.scd1_trx_acq',
    'ods_alpha.scd1_trx_int',
    'ods_alpha.scd1_base24_fiids',
    'ods.scd1_z_r2_ip_merchants',
    'ods.scd1_z_r2_tariff_tune',
    'ods.scd1_z_r2_tariff_fix',
    'ods.scd1_z_cl_corp',
    'ods.scd1_z_depart',
    'ods.scd1_z_branch',
    'ods.scd1_z_r2_tariff_plan'
]

invalidate_ok = 0
invalidate_failed = []
with imp:
    for t in invalidate_tables:
        try:
            imp.execute(f'invalidate metadata {t}')
            invalidate_ok += 1
        except Exception as e:
            invalidate_failed.append((t, str(e)))

print(f'Invalidate успешно: {invalidate_ok}/{len(invalidate_tables)}')
if invalidate_failed:
    print(f'Invalidate с ошибкой: {len(invalidate_failed)} (продолжаем выполнение)')
    pd.DataFrame(invalidate_failed, columns=['table_name', 'error_message']).head(20)

## Сборка витрины по MVP-атрибутам

Последовательный SQL-pipeline от SA-периметра до финальной витрины.

In [ ]:
# Переход к расчетным шагам сборки витрины.

### 4.1) Функции нормализации

Отдельная ячейка с функциями для очистки и приведения данных.

In [ ]:
def normalize_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'[^0-9]', '', str(v).strip())
    return s or None


def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    return s if s else None


def to_decimal_or_none(v):
    if pd.isna(v):
        return None
    if isinstance(v, Decimal):
        return v
    try:
        return Decimal(str(v).strip().replace(',', '.'))
    except (InvalidOperation, ValueError):
        return None

### 4.2) SA-периметр (базовый слой)

Берем из Озера клиентов и договоры, активные в выбранном месяце, и получаем базовый периметр для последующего обогащения.

In [ ]:
sql_sa_perimeter = f"""
select distinct
  cast(a.n_agr as string) as n_agr,
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(c.c_cmp_name as string) as company_name
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where upper(trim(cast(a.acq_class as string))) = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
  and c.c_inn is not null
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    sa_df = imp.fetch(sql_sa_perimeter)

if sa_df is None:
    sa_df = pd.DataFrame()

if not sa_df.empty:
    sa_df['inn'] = sa_df['inn'].map(normalize_inn)
    sa_df['contract_number'] = sa_df['contract_number'].map(normalize_contract)

sa_rows_cnt = len(sa_df)
sa_inn_unique_cnt = sa_df['inn'].dropna().astype(str).nunique() if 'inn' in sa_df.columns else 0
print(f'Записей в SA-периметре: {sa_rows_cnt:,}')
print(f'Уникальных ИНН в SA-периметре: {sa_inn_unique_cnt:,}')

sa_df.head(5)

### 4.3) Маппинг INN -> CDI

Для ИНН из периметра ищем актуальный OCRM `row_id` и связываем его с `cdi_id` через `cdiul.ext_id_org`.

In [ ]:
inn_values = sorted([x for x in sa_df.get('inn', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
inn_sql_list = ', '.join([f"'{x}'" for x in inn_values]) if inn_values else "''"

sql_cdi = f"""
with ocrm_current as (
  select
    regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
    cast(soe.row_id as string) as row_id,
    trim(cast(soe.x_area_resp as string)) as x_area_resp_norm,
    case
      when soe.x_area_resp like 'ДММБ%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ(ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ (ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДСБ%' then 'ДМСБ'
      when soe.x_area_resp like 'ДКБ%' then 'ДКБ'
      when soe.x_area_resp like 'ДРПА%' then 'ДРПА'
      when soe.x_area_resp like 'ДРА%' then 'ДРПА'
      when lower(soe.x_area_resp) like 'не под%' then 'Не подлежит сегментации'
      when nullif(trim(cast(soe.x_area_resp as string)), '') is null then null
      else null
    end as ssp_ocrm,
    row_number() over (
      partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
      order by cast(soe.created as timestamp) desc,
               cast(soe.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext soe
  where regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') in ({inn_sql_list})
    and coalesce(soe.x_removed_flg, 'N') = 'N'
    and coalesce(soe.x_duplicate_flg, 'N') = 'N'
),
ocrm_one as (
  select inn, row_id, ssp_ocrm, x_area_resp_norm
  from ocrm_current
  where rn = 1
)
select
  o.inn,
  o.ssp_ocrm,
  o.x_area_resp_norm,
  cast(e.party_id as string) as cdi_id
from ocrm_one o
left join cdiul.ext_id_org e
  on cast(e.cmo_ext_party_source_id as string) = o.row_id
 and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cdi_map_df = imp.fetch(sql_cdi)

if cdi_map_df is None:
    cdi_map_df = pd.DataFrame(columns=['inn', 'ssp_ocrm', 'x_area_resp_norm', 'cdi_id'])

if not cdi_map_df.empty:
    cdi_map_df['inn'] = cdi_map_df['inn'].map(normalize_inn)
    cdi_map_df['cdi_id'] = cdi_map_df['cdi_id'].astype(str)

    # Финальный контроль канонических сегментов
    allowed_ssp_values = {'ДМ', 'ДМСБ', 'ДКБ', 'ДРПА', 'Не подлежит сегментации'}
    cdi_map_df['ssp_ocrm'] = cdi_map_df['ssp_ocrm'].where(cdi_map_df['ssp_ocrm'].isin(allowed_ssp_values), None)

    cdi_map_df = cdi_map_df.drop_duplicates(subset=['inn'], keep='first')

cdi_map_df.head(5)

### 4.4) Маппинг CDI -> CFT

Для найденных `cdi_id` получаем соответствующие `cft_id`, чтобы перейти к операционным и финансовым метрикам R2.

In [ ]:
cdi_values = sorted([x for x in cdi_map_df.get('cdi_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
cdi_sql_list = ', '.join([f"'{x}'" for x in cdi_values]) if cdi_values else "''"

sql_cft = f"""
select
  cast(e.party_id as string) as cdi_id,
  cast(e.cmo_ext_party_source_id as string) as cft_id
from cdiul.ext_id_org e
where cast(e.party_id as string) in ({cdi_sql_list})
  and upper(cast(e.cmo_ext_source_system as string)) like 'CFT%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cft_map_df = imp.fetch(sql_cft)

if cft_map_df is None:
    cft_map_df = pd.DataFrame(columns=['cdi_id', 'cft_id'])

if not cft_map_df.empty:
    cft_map_df['cdi_id'] = cft_map_df['cdi_id'].astype(str)
    cft_map_df['cft_id'] = cft_map_df['cft_id'].astype(str)
    cft_map_df = cft_map_df.drop_duplicates(subset=['cdi_id'], keep='first')

cft_map_df.head(5)

### 4.5) Операционные и транзакционные атрибуты

Считаем метрики на уровне компании (`retl_cnt`, `term_cnt`, `amortization`) и договора (`trx_cnt`, `trx_sum`, `commission_from_ops`, `int_component`).

In [ ]:
# 4.5.1) Company-level metrics (retl_cnt, term_cnt, active_term_cnt, amortization)
cmp_ids = sorted(sa_df.get('n_cmp_client', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
cmp_in = ', '.join([f"'{x}'" for x in cmp_ids]) if cmp_ids else "''"

n_agrs = sorted(sa_df.get('n_agr', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
agr_in = ', '.join([f"'{x}'" for x in n_agrs]) if n_agrs else "''"

sql_cmp = f"""
with m as (
  select
    cast(n_cmp as string) as n_cmp_client,
    cast(c_nmrc as string) as c_nmrc,
    cast(n_mcc as string) as n_mcc
  from ods_alpha.scd1_merchants
  where cast(n_cmp as string) in ({cmp_in})
    and c_nmrc is not null
),
retl as (
  select n_cmp_client,
         count(distinct c_nmrc) as retl_cnt
  from m
  group by n_cmp_client
),
term as (
  select m.n_cmp_client,
         count(distinct t.c_nter) as term_cnt
  from m
  join ods_alpha.scd1_pos_terminals t
    on t.c_nmrc = m.c_nmrc and t.c_nter is not null
  group by m.n_cmp_client
),
active_term as (
  select
    cast(sa.n_cmp_client as string) as n_cmp_client,
    count(distinct cast(t.c_nter as string)) as active_term_cnt
  from ods_alpha.scd1_trx t
  join (
    select distinct
      cast(n_agr as string) as n_agr,
      cast(n_cmp_client as string) as n_cmp_client
    from ods_alpha.scd1_agreements
    where cast(n_agr as string) in ({agr_in})
  ) sa on cast(sa.n_agr as string) = cast(t.n_agr as string)
  left join ods_alpha.scd1_base24_fiids fa
    on cast(fa.c_fiid as string) = cast(t.c_fiid_acq as string)
  where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
    and t.c_nter is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
    and coalesce(t.cf_trx_stat, '') <> 'R'
    and coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    and coalesce(cast(t.n_amt_src as double), 0.0) > 1
  group by cast(sa.n_cmp_client as string)
),
amort as (
  select m.n_cmp_client,
         sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization
  from m
  join ods_alpha.scd1_pos_terminals t
    on t.c_nmrc = m.c_nmrc and t.c_nter is not null
  left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
    on cast(am.c_nter as string) = cast(t.c_nter as string)
  group by m.n_cmp_client
)
select
  coalesce(r.n_cmp_client, t.n_cmp_client, at.n_cmp_client, a.n_cmp_client) as n_cmp_client,
  r.retl_cnt,
  t.term_cnt,
  at.active_term_cnt,
  a.amortization
from retl r
full outer join term t on t.n_cmp_client = r.n_cmp_client
full outer join active_term at on at.n_cmp_client = coalesce(r.n_cmp_client, t.n_cmp_client)
full outer join amort a on a.n_cmp_client = coalesce(r.n_cmp_client, t.n_cmp_client, at.n_cmp_client)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cmp_df = imp.fetch(sql_cmp)

if cmp_df is None:
    cmp_df = pd.DataFrame(columns=['n_cmp_client', 'retl_cnt', 'term_cnt', 'active_term_cnt', 'amortization'])

if not cmp_df.empty:
    cmp_df['n_cmp_client'] = cmp_df['n_cmp_client'].astype(str)

cmp_df.head(5)

### 4.5.2) TRX-метрики по договору

Отдельный шаг расчета транзакционных метрик по `n_agr` (проще отладка при ошибках в `trx`-логике).

In [ ]:
# 4.5.2) Transaction metrics by agreement (trx_cnt, trx_sum, commission_from_ops, int_component)
n_agrs = sorted(sa_df.get('n_agr', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
agr_in = ', '.join([f"'{x}'" for x in n_agrs]) if n_agrs else "''"

sql_trx = f"""
with trx_base as (
  select
    cast(t.n_trx as string) as n_trx,
    cast(t.n_amt_src as double) as n_amt_src
  from ods_alpha.scd1_trx t
  left join ods_alpha.scd1_base24_fiids fa
    on cast(fa.c_fiid as string) = cast(t.c_fiid_acq as string)
  where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
    and t.c_nter is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
    and coalesce(t.cf_trx_stat, '') <> 'R'
    and coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
),
ta as (
  select
    cast(n_trx as string) as n_trx,
    cast(n_agr as string) as n_agr,
    coalesce(cast(n_amt_tax as double), 0.0) as n_amt_tax
  from ods_alpha.scd1_trx_acq
  where cast(n_agr as string) in ({agr_in})
),
tj as (
  select
    ta.n_agr,
    ta.n_trx,
    tb.n_amt_src,
    ta.n_amt_tax
  from ta
  join trx_base tb on tb.n_trx = ta.n_trx
)
select
  tj.n_agr,
  count(distinct tj.n_trx) as trx_cnt,
  sum(tj.n_amt_src) as trx_sum,
  sum(tj.n_amt_tax) as commission_from_ops,
  sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as int_component
from tj
left join ods_alpha.scd1_trx_int i on cast(i.n_trx as string) = tj.n_trx
group by tj.n_agr
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    trx_df = imp.fetch(sql_trx)

if trx_df is None:
    trx_df = pd.DataFrame(columns=['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component'])

if not trx_df.empty:
    trx_df['n_agr'] = trx_df['n_agr'].astype(str)

trx_df.head(5)

### 4.5.3) R2-атрибуты (оргструктура и тариф)

Отдельный шаг R2, чтобы ошибки по орг/тарифным полям не маскировались в общей секции.

In [ ]:
cft_values = sorted([x for x in cft_map_df.get('cft_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
cft_sql_list = ', '.join([f"'{x}'" for x in cft_values]) if cft_values else "''"

sql_r2 = f"""
with r2 as (
  select
    cast(m.id as string) as r2_id,
    cast(m.c_cl_org as string) as cft_id,
    cast(m.c_depart as string) as c_depart,
    cast(m.c_tariff_plan as string) as c_tariff_plan
  from ods.scd1_z_r2_ip_merchants m
  where cast(m.c_cl_org as string) in ({cft_sql_list})
),
fix as (
  select
    cast(tt.c_tariff_plan as string) as c_tariff_plan,
    max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
  from ods.scd1_z_r2_tariff_tune tt
  left join ods.scd1_z_r2_tariff_fix tf on tt.c_tariff = tf.id
  group by cast(tt.c_tariff_plan as string)
)
select
  r2.r2_id,
  r2.cft_id,
  cast(corp.c_register_gos_reg_num_rec as string) as ogrn,
  cast(dep.c_name as string) as vsp_name,
  cast(dep.c_num as string) as vsp_code,
  case
    when br.c_shortlabel is null then null
    when upper(cast(br.c_shortlabel as string)) like '%РФ%'
      then regexp_extract(cast(br.c_shortlabel as string), '^(.*?РФ)', 1)
    else cast(br.c_shortlabel as string)
  end as filial_rf,
  cast(tp.c_name as string) as tariff_name,
  cast(fix.commission_monthly_fix as decimal(18,2)) as commission_monthly
from r2
left join ods.scd1_z_cl_corp corp on cast(corp.id as string) = r2.cft_id
left join ods.scd1_z_depart dep on cast(dep.id as string) = r2.c_depart
left join ods.scd1_z_branch br on cast(br.id as string) = cast(dep.c_filial as string)
left join ods.scd1_z_r2_tariff_plan tp on cast(tp.id as string) = r2.c_tariff_plan
left join fix on fix.c_tariff_plan = r2.c_tariff_plan
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    r2_df = imp.fetch(sql_r2)

if r2_df is None:
    r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name', 'commission_monthly'])

if not r2_df.empty:
    for c in ['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name']:
        r2_df[c] = r2_df[c].astype(str)

    r2_rows_before = len(r2_df)
    r2_df = r2_df.drop_duplicates(subset=['cft_id'], keep='first')
    r2_rows_after = len(r2_df)
    print(f'R2 строк до дедупликации по cft_id: {r2_rows_before:,}')
    print(f'R2 строк после дедупликации по cft_id: {r2_rows_after:,}')

r2_df.head(5)

### 4.6) Финальная сборка MVP-витрины

Объединяем SA, CDI, CFT, CMP, TRX и R2 в итоговый датасет.

In [ ]:
base_df = sa_df.copy()
print(f'Строк на старте (SA): {len(base_df):,}')

if not cdi_map_df.empty and not base_df.empty:
    before_rows = len(base_df)
    base_df = base_df.merge(cdi_map_df[['inn', 'cdi_id', 'ssp_ocrm']], on='inn', how='left')
    print(f'После merge INN->CDI: {before_rows:,} -> {len(base_df):,}')
else:
    base_df['cdi_id'] = None
    base_df['ssp_ocrm'] = None

if not cft_map_df.empty and not base_df.empty:
    before_rows = len(base_df)
    base_df = base_df.merge(cft_map_df[['cdi_id', 'cft_id']], on='cdi_id', how='left')
    print(f'После merge CDI->CFT: {before_rows:,} -> {len(base_df):,}')
else:
    base_df['cft_id'] = None

if not r2_df.empty and not base_df.empty:
    before_rows = len(base_df)
    base_df = base_df.merge(r2_df, on='cft_id', how='left')
    print(f'После merge CFT->R2: {before_rows:,} -> {len(base_df):,}')
else:
    for col in ['r2_id', 'commission_monthly', 'ogrn', 'filial_rf', 'vsp_name', 'vsp_code', 'tariff_name']:
        base_df[col] = None

if not cmp_df.empty and not base_df.empty:
    before_rows = len(base_df)
    base_df = base_df.merge(cmp_df[['n_cmp_client', 'retl_cnt', 'term_cnt', 'active_term_cnt', 'amortization']], on='n_cmp_client', how='left')
    print(f'После merge company metrics: {before_rows:,} -> {len(base_df):,}')
else:
    for col in ['retl_cnt', 'term_cnt', 'active_term_cnt', 'amortization']:
        base_df[col] = None

if not trx_df.empty and not base_df.empty:
    before_rows = len(base_df)
    base_df = base_df.merge(trx_df[['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component']], on='n_agr', how='left')
    print(f'После merge TRX metrics: {before_rows:,} -> {len(base_df):,}')
else:
    for col in ['trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component']:
        base_df[col] = None

if 'agr_id' not in base_df.columns:
    base_df['agr_id'] = None
if 'r2_id' not in base_df.columns:
    base_df['r2_id'] = None

agr_id_before_fill = base_df['agr_id'].notna() & (base_df['agr_id'].astype(str).str.strip() != '')
base_df['agr_id_source'] = 'sa'
fallback_mask = (~agr_id_before_fill) & base_df['r2_id'].notna() & (base_df['r2_id'].astype(str).str.strip() != '')
base_df.loc[fallback_mask, 'agr_id'] = base_df.loc[fallback_mask, 'r2_id']
base_df.loc[fallback_mask, 'agr_id_source'] = 'r2_fallback'

agr_id_after_fill = base_df['agr_id'].notna() & (base_df['agr_id'].astype(str).str.strip() != '')
agr_fill_before_pct = round(100.0 * agr_id_before_fill.mean(), 2) if len(base_df) else 0.0
agr_fill_after_pct = round(100.0 * agr_id_after_fill.mean(), 2) if len(base_df) else 0.0
agr_fallback_rows = int(fallback_mask.sum())
print(f'Заполняемость agr_id до fallback: {agr_fill_before_pct}%')
print(f'Заполняемость agr_id после fallback: {agr_fill_after_pct}%')
print(f'Строк, где применен fallback agr_id <- r2_id: {agr_fallback_rows:,}')

commission_from_ops_num = pd.to_numeric(base_df.get('commission_from_ops'), errors='coerce').fillna(0)
commission_monthly_num = pd.to_numeric(base_df.get('commission_monthly'), errors='coerce').fillna(0)
int_component_num = pd.to_numeric(base_df.get('int_component'), errors='coerce').fillna(0)
amortization_num = pd.to_numeric(base_df.get('amortization'), errors='coerce').fillna(0)

base_df['commission_total'] = commission_from_ops_num + commission_monthly_num

term_cnt_num = pd.to_numeric(base_df.get('term_cnt'), errors='coerce')
retl_cnt_num = pd.to_numeric(base_df.get('retl_cnt'), errors='coerce')
aur_units = term_cnt_num.where(term_cnt_num.notna() & (term_cnt_num > 0), retl_cnt_num)
base_df['aur'] = aur_units * 1926

aur_num = pd.to_numeric(base_df.get('aur'), errors='coerce').fillna(0)
base_df['chod'] = base_df['commission_total'] - int_component_num
base_df['fin_result'] = base_df['chod'] - aur_num - amortization_num
base_df['report_month'] = report_month_label
base_df['snapshot_month_start'] = snapshot_month_start

mvp_columns = [
    'report_month', 'snapshot_month_start', 'inn', 'company_name',
    'agr_id', 'n_agr', 'contract_number', 'd_valid_from', 'd_valid_to',
    'cdi_id', 'ssp_ocrm', 'ogrn', 'filial_rf', 'vsp_name', 'vsp_code', 'tariff_name',
    'retl_cnt', 'term_cnt', 'active_term_cnt', 'trx_cnt', 'trx_sum',
    'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total',
    'aur', 'amortization', 'chod', 'fin_result'
]

for col in mvp_columns:
    if col not in base_df.columns:
        base_df[col] = None

for col in ['trx_sum', 'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total', 'aur', 'amortization', 'chod', 'fin_result']:
    if col in base_df.columns:
        base_df[col] = base_df[col].map(to_decimal_or_none)

for col in ['retl_cnt', 'term_cnt', 'active_term_cnt', 'trx_cnt']:
    if col in base_df.columns:
        base_df[col] = pd.to_numeric(base_df[col], errors='coerce').astype('Int64')

final_df = base_df[mvp_columns].copy()

for forbidden_col in ['n_cmp_client', 'key_tier', 'dq_block_reason', 'r2_id', 'agr_id_source']:
    if forbidden_col in final_df.columns:
        final_df = final_df.drop(columns=[forbidden_col])

print(f'Собрано строк: {len(final_df):,}')
print(f'Атрибутов в финале: {len(final_df.columns)}')

## 5) Проверка заполняемости полей витрины

Контроль заполненности каждого атрибута финальной витрины для поиска проблемных мест.

In [ ]:
row_cnt = len(final_df)

field_fill_stats = []
for col in final_df.columns:
    non_null_cnt = int(final_df[col].notna().sum())
    null_cnt = int(row_cnt - non_null_cnt)
    fill_rate_pct = round((non_null_cnt / row_cnt) * 100, 2) if row_cnt else 0.0
    field_fill_stats.append({
        'field_name': col,
        'non_null_cnt': non_null_cnt,
        'null_cnt': null_cnt,
        'fill_rate_pct': fill_rate_pct
    })

fill_stats_df = pd.DataFrame(field_fill_stats).sort_values(['fill_rate_pct', 'field_name'], ascending=[True, True]).reset_index(drop=True)

print(f'Строк в финальной витрине: {row_cnt:,}')
fill_stats_df

## 6) Предпросмотр финальной витрины

Показываем первые 5 строк перед выгрузкой.

In [ ]:
final_df.head(5)

## 7) Выгрузка в CSV

Сохраняем результат в указанный путь.
Для загрузки в Greenplum/DRP используем CSV (а не Excel): CSV проще для массовой загрузки, стабильнее для автоматизации и корректнее для типизации колонок.

In [ ]:
output_path = Path(output_csv_path)
output_path.parent.mkdir(parents=True, exist_ok=True)

final_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'CSV сохранен: {output_path.resolve()}')
print(f'Строк: {len(final_df):,}')